In [3]:
import json
from pathlib import Path

import cv2
import numpy as np
from tqdm import tqdm


# Match your datasets.py folder setup
DATA_FOLDER = "../Data200x200_withInfo_Deterministic/Data200x200_withInfo_Deterministic/"
BINARY_FOLDER = "../Data200x200_withInfo_Deterministic/Data200x200_withInfo_Deterministic/"

TRAIN_SIMS_PATH = "../train_sims.npy"

OUT_PATH = "dataset_priors.json"


def load_all(sim, step, folder=DATA_FOLDER):
    """
    Returns [3, H, W] normalized to roughly [-1, 1].
    Channel order:
        0 = K
        1 = P
        2 = phi
    """
    path = Path(folder) / f"Sim-{sim}-Step-{step}.png"
    img = cv2.imread(str(path))

    if img is None:
        raise FileNotFoundError(f"Could not read {path}")

    return (img.transpose(2, 0, 1).astype(np.float32) - 128) / 128


def compute_priors(
    sims,
    steps=range(0, 200),
    folder=DATA_FOLDER,
    low_k_threshold=-0.5,
):
    """
    Computes dataset-level statistics.

    These are not sample-specific targets.
    They are general priors learned from the training distribution.
    """

    channel_sum = np.zeros(3, dtype=np.float64)
    channel_sumsq = np.zeros(3, dtype=np.float64)
    pixel_count = 0

    low_k_area_values = []

    for sim in tqdm(sims, desc="Computing priors"):
        for step in steps:
            x = load_all(sim, step, folder=folder)  # [3, H, W]

            C, H, W = x.shape
            flat = x.reshape(C, -1)

            channel_sum += flat.sum(axis=1)
            channel_sumsq += (flat ** 2).sum(axis=1)
            pixel_count += H * W

            k = x[0]
            low_k_area = (k < low_k_threshold).mean()
            low_k_area_values.append(float(low_k_area))

    channel_mean = channel_sum / pixel_count
    channel_var = channel_sumsq / pixel_count - channel_mean ** 2
    channel_std = np.sqrt(np.maximum(channel_var, 0.0))

    priors = {
        "channel_order": ["K", "P", "phi"],
        "channel_mean": channel_mean.tolist(),
        "channel_std": channel_std.tolist(),

        "low_k_threshold": low_k_threshold,
        "low_k_area_mean": float(np.mean(low_k_area_values)),
        "low_k_area_std": float(np.std(low_k_area_values)),

        "num_sims": int(len(sims)),
        "num_steps": int(len(list(steps))),
        "folder": folder,
    }

    return priors


if __name__ == "__main__":
    train_sims = np.load(TRAIN_SIMS_PATH)

    # Optional: match your training cutoff
    train_sims = train_sims[train_sims < 500]

    priors = compute_priors(
        sims=train_sims,
        steps=range(1, 200),
        folder=DATA_FOLDER,
        low_k_threshold=-0.5,
    )

    with open(OUT_PATH, "w") as f:
        json.dump(priors, f, indent=2)

    print(json.dumps(priors, indent=2))
    print(f"Saved priors to {OUT_PATH}")

Computing priors: 100%|██████████| 307/307 [02:28<00:00,  2.07it/s]

{
  "channel_order": [
    "K",
    "P",
    "phi"
  ],
  "channel_mean": [
    -0.18784285351201244,
    -0.6110295218298076,
    -0.9120737134049216
  ],
  "channel_std": [
    0.2515213950708161,
    0.3208698303289577,
    0.19905961457025312
  ],
  "low_k_threshold": -0.5,
  "low_k_area_mean": 0.08575309937308692,
  "low_k_area_std": 0.04581761097715663,
  "num_sims": 307,
  "num_steps": 199,
  "folder": "../Data200x200_withInfo_Deterministic/Data200x200_withInfo_Deterministic/"
}
Saved priors to dataset_priors.json
